# Práctica 3. Detección y reconocimiento de formas

---

**Autores:**
- Alba Ramos Quintana
- Néstor Jesús Henríquez Medina

**Descripción:** Resolución de las tareas de la Práctica 3, incluyendo la identificación y clasificación de monedas para calcular cantidades de dinero, el procesamiento de imágenes de microplásticos mediante técnicas de umbralizado y el desarrollo de un clasificador basado en características geométricas y visuales.

---

## Tarea 1: Detección y reconocimiento de monedas

### Tarea 1.1: Cálculo del dinero en una imagen

- Determina la cantidad total de dinero presente en una imagen de monedas a partir de sus dimensiones. Para ello, establece una relación entre píxeles y milímetros utilizando como referencia una moneda de valor conocido y compara los diámetros medidos con las dimensiones reales de las monedas de euro.

In [ ]:
%matplotlib qt

import cv2  
import numpy as np
import matplotlib.pyplot as plt
import csv

diametro_monedas = {
    2.00: 25.75,
    1.00: 23.25,
    0.50: 24.25,
    0.20: 22.25,
    0.10: 19.75,
    0.05: 21.25,
    0.02: 18.75,
    0.01: 16.25
}

def clasificar_moneda(diametro_medido, dicccionario):
    """
    Adivina de qué moneda se trata comparando su tamaño medido con las medidas oficiales.

    Calcula la diferencia entre el diámetro que hemos sacado en milímetros y los 
    tamaños reales del diccionario y escoge la opción que más se acerque.
    """
    moneda = None
    min_diferencia = float('inf')

    for valor, diametro in dicccionario.items():
        dif = abs(diametro - diametro_medido)
        if dif < min_diferencia:
            min_diferencia = dif
            moneda = valor
    return moneda

img = cv2.imread('Monedas.jpg') 
#img = cv2.imread('monedas2.jpg')
print(img.shape)

img_gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
cv2.imwrite('tarea_1/1_original_gris.jpg', img_gris)

img_suavizada = cv2.GaussianBlur(img_gris, (5, 5), 0)
cv2.imwrite('tarea_1/2_suavizada_gaussiana.jpg', img_suavizada)

img_mediana = cv2.medianBlur(img_gris, 5)
cv2.imwrite('tarea_1/2_suavizada_mediana.jpg', img_mediana)

th,img_th = cv2.threshold(img_gris,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
print('Umbral Otsu ', th)
cv2.imwrite('tarea_1/3_binarizada.jpg', img_th)

contornos, _ = cv2.findContours(
    img_th, 
    cv2.RETR_EXTERNAL , 
    cv2.CHAIN_APPROX_SIMPLE
    )

print("Hay un total de", len(contornos), "contonos detectados")


# Fuente: https://hetpro-store.com/TUTORIALES/opencv-findcontours/?srsltid=AU7gw4Vh8qvHAzdz3LKVhhBp38z6-qQP5ekuIxHiYYR8xJRL_PztFQNX
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2.drawContours(img_rgb, contornos, -1, (0,0,255), 3)

plt.plot()
plt.axis("off")
plt.imshow(img_rgb) 

# Fuente: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.ginput.html
# Fuente: búsqueda en Google - "ginput() en ipynb"
# pip install PyQt5
pts = plt.ginput(1)

if len(pts) > 0:
    clic_x, clic_y = pts[0]
    print(f"Coordenada seleccionada: X= {clic_x:.2f}, Y= {clic_y:.2f}")

    contorno_ref = None
    for contorno in contornos:
        # Fuente: búsqueda en Google - "Averiguar punto dentro de un contorno con findContours()"
        ref = cv2.pointPolygonTest(contorno, (clic_x, clic_y), False)
        #print(ref)
        if ref >= 0:
            contorno_ref = contorno
            print("Esto es una moneda")
            break

    # Fuente: https://docs.opencv.org/4.13.0/dd/d49/tutorial_py_contour_features.html
    if contorno_ref is not None:
        _, radio_ref = cv2.minEnclosingCircle(contorno_ref)
        diametro_ref = radio_ref*2
        escala = diametro_monedas[1.00] / diametro_ref
        #print(escala)

        cv2.drawContours(img_rgb, [contorno_ref], -1, (0,255,0), 5)
        plt.imshow(img_rgb)
        plt.draw()

        total = 0
        diametro_min_px = 15/escala # min 15 mm
        for contorno in contornos:
            centro, radio = cv2.minEnclosingCircle(contorno)
            diametro_px = radio * 2
            
            # Si es muy pequeño, pasamos al siguiente contorno
            if diametro_px < diametro_min_px:
                continue
                
            diametro_mm = diametro_px * escala
            valor_moneda = clasificar_moneda(diametro_mm, diametro_monedas)
            
            total += valor_moneda
            
            texto = f"{valor_moneda}€"
            x, y = int(centro[0]), int(centro[1])
            cv2.putText(img_rgb, texto, (x - 20, y + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
        
        total_text = f"TOTAL = {total:.2f}€"
        tam_text, _ = cv2.getTextSize(total_text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 2)

        cv2.putText(img_rgb, total_text, (int((img.shape[1] - tam_text[0]) / 2), img.shape[0]), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2, cv2.LINE_AA)

        print("\n==============")
        print(total_text)
        print("==============")

        img_final_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        cv2.imwrite('tarea_1/4_resultado.jpg', img_final_bgr)
        plt.imshow(img_rgb)
        plt.draw()


(938, 473, 3)
Umbral Otsu  204.0
Hay un total de 50 contonos detectados
Coordenada seleccionada: X= 93.88, Y= 360.89
Esto es una moneda

TOTAL = 3.88€


---

### Tarea 1.2: Demostrador interactivo en tiempo real

- Diseña un demostrador que utilice la cámara para detectar las monedas presentes en una escena y determinar la cantidad de dinero total, sin utilizar modelos de aprendizaje profundo.

#### Análisis y Evolución

Para resolver el ejercicio, se ha seguido un proceso de desarrollo iterativo. Tras evaluar múltiples aproximaciones experimentales, se han seleccionado **dos versiones clave** que muestran la evolución progresiva del sistema ante problemas reales del entorno (como el ruido de la cámara, las sombras, la iluminación variable y el solape entre objetos).

---

##### 1. Versión Inicial: Detección Geométrica por Contornos y Binarización
Esta primera aproximación se centró en la segmentación mediante umbralización adaptativa (`cv2.adaptiveThreshold`), filtrado bilateral y la extracción de contornos (`cv2.findContours`). 
* **Funcionamiento:** Tras calibrar la escala haciendo clic en una moneda de referencia (1€), el sistema calcula la compacidad geométrica y el radio mínimo envolvente para filtrar formas circulares.
* **Limitaciones observadas:** Aunque el recuento es estable cuando las condiciones son perfectas, este método colapsa en escenarios reales. Si dos monedas se rozan o solapan, las fusiona en una única máscara, arruinando el cálculo de contornos. Asimismo, los reflejos especulares de las luces de la habitación generan huecos internos que rompen la binarización.

In [ ]:
import cv2
import numpy as np

diametro_monedas = {
    2.00: 25.75,
    1.00: 23.25,
    0.50: 24.25,
    0.20: 22.25,
    0.10: 19.75,
    0.05: 21.35,
    0.02: 18.75,
    0.01: 16.25
}

def clasificar_moneda(diametro_medido, diccionario):
    """
        Adivina de qué moneda se trata comparando su tamaño medido con las medidas oficiales.
    
        Calcula la diferencia entre el diámetro que hemos sacado en milímetros y los 
        tamaños reales del diccionario y escoge la opción que más se acerque.
    """
    moneda = None
    min_diferencia = float('inf')
    for valor, diametro in diccionario.items():
        dif = abs(diametro - diametro_medido)
        if dif < min_diferencia:
            min_diferencia = dif
            moneda = valor
    return moneda

escala = None
punto_clic = None
invertir_mascara = False

# Fuente: https://www.geeksforgeeks.org/python/click-response-on-video-output-using-events-in-opencv-python/
def detectar_clic(evento, x, y, flags, param):
    """Guarda las coordenadas al hacer clic izquierdo."""
    global punto_clic
    if evento == cv2.EVENT_LBUTTONDOWN:
        punto_clic = (x, y)

vid = cv2.VideoCapture(1)

cv2.namedWindow('Cam')
cv2.setMouseCallback('Cam', detectar_clic)

# Fuente: https://www.geeksforgeeks.org/python/clahe-histogram-eqalization-opencv/
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

while True:
    ret, frame = vid.read()

    if ret:
        gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        gris_clahe = clahe.apply(gris)
        
        # Fuente: https://www.geeksforgeeks.org/python/python-bilateral-filtering/
        gris_suavizado = cv2.bilateralFilter(gris_clahe, d=15, sigmaColor=85, sigmaSpace=85)
        
        if invertir_mascara:
            modo_umbral = cv2.THRESH_BINARY_INV
        else:
            modo_umbral = cv2.THRESH_BINARY
            
        img_th = cv2.adaptiveThreshold(gris_suavizado, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                       modo_umbral, 99, 25)

        kernel_cierre = np.ones((7,7), np.uint8)
        kernel_apertura = np.ones((5, 5), np.uint8)

        # Fuente: https://www.luisllamas.es/opencv-operaciones-morfologicas-erosion-dilatacion/
        img_th = cv2.morphologyEx(img_th, cv2.MORPH_CLOSE, kernel_cierre, iterations=2)
        img_th = cv2.morphologyEx(img_th, cv2.MORPH_OPEN, kernel_apertura, iterations=1)

        contornos, _ = cv2.findContours(img_th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Calibración al hacer clic
        if punto_clic is not None and escala is None:
            for contorno in contornos:
                if cv2.pointPolygonTest(contorno, punto_clic, False) >= 0:
                    _, radio_ref = cv2.minEnclosingCircle(contorno)
                    if radio_ref > 0:
                        diametro_ref = radio_ref * 2
                        escala = diametro_monedas[1.00] / diametro_ref
                        print(f"Calibrada escala: {escala:.4f}")
            punto_clic = None

        # Detección y Suma
        if escala is None:
            cv2.putText(frame, "Haz clic en la de 1€ (R: Invertir máscara)", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        else:
            total = 0.0
            for c in contornos:
                area = cv2.contourArea(c)
                if area < 150: # Ignorar manchas pequeñas
                    continue
                    
                perimetro = cv2.arcLength(c, True)
                if perimetro == 0:
                    continue
                
                compacidad = (perimetro ** 2) / area
                
                # Filtrar solo objetos circulares
                if 12.0 < compacidad < 30.0:
                    centro, radio = cv2.minEnclosingCircle(c)
                    diametro_px = radio * 2
                    diametro_mm = diametro_px * escala
                    
                    if diametro_mm > 15.0:
                        valor = clasificar_moneda(diametro_mm, diametro_monedas)
                        total += valor
                        
                        cv2.circle(frame, (int(centro[0]), int(centro[1])), int(radio), (0, 255, 0), 2)
                        x_txt, y_txt = int(centro[0]), int(centro[1])
                        cv2.putText(frame, f"{valor}E", (x_txt - 20, y_txt + 5), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
            
            texto_total = f"TOTAL = {total:.2f} €"
            cv2.putText(frame, texto_total, (20, frame.shape[0] - 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)

        cv2.imshow('Cam', frame)
        cv2.imshow('Mascara', img_th)

    tecla = cv2.waitKey(20) & 0xFF
    if tecla == 27: 
        break
    elif tecla == ord('r') or tecla == ord('R'): 
        invertir_mascara = not invertir_mascara
        escala = None 
        print("Máscara invertida")

vid.release()
cv2.destroyAllWindows()

#### 2. Versión Definitiva: Transformada de Hough + Análisis HSV + Estabilización Temporal
Para superar las debilidades de la binarización estática y los solapes, se rediseñó por completo el núcleo del demostrador integrando la **Transformada de Hough para gradientes circulares (`cv2.HoughCircles`)**, capaz de aislar y completar geometrías redondas incluso con objetos tangentes o reflejos intensos.

- **Clasificación Segregada por Color (HSV):** Debido a que hay monedas casi idénticas, que las diferencia apenas 1 mm, el sistema sufría errores por la perspectiva de la cámara. Para solucionarlo, el algoritmo extrae máscaras concéntricas del centro (40%) y del anillo exterior (85%), evaluando la diferencia de saturación para aislar con precisión las monedas bimetálicas de las doradas y cobrizas, eliminando la competencia directa por tamaño.
- **Filtro de Estabilización Temporal:** Para mitigar el "ruido del sensor" de la cámara, se implementó una "ventana deslizante" de memoria de 15 fotogramas aplicando la moda sobre totales redondeados (`round`). Esto absorbe los picos de error y congela la cifra final en pantalla sin las molestas vibraciones del vídeo en directo.

> _**Nota sobre la estabilidad en tiempo real:** Aunque este código mejora los resultados frente a la detección por contornos y reduce drásticamente las oscilaciones, el sistema en vídeo directo no alcanza una estabilidad absoluta. Las micro-variaciones lumínicas del entorno, el parpadeo del sensor de la cámara web y los diminutos cambios en el ángulo de perspectiva siguen provocando fluctuaciones puntuales en fotogramas aislados. No obstante, representa un equilibrio medianamente funcional dentro de las restricciones de la visión por computador clásica sin aprendizaje profundo._

In [ ]:
import cv2
import numpy as np

monedas_cobre = {0.05: 21.25, 0.02: 18.75, 0.01: 16.25}
monedas_doradas = {0.50: 24.25, 0.20: 22.25, 0.10: 19.75}
monedas_bimetal = {2.00: 25.75, 1.00: 23.25}

def clasificar_moneda_hsv(diametro_medido, hsv_centro, hsv_anillo):
    """
        Clasifica una moneda según su tamaño en milímetros y su perfil de color HSV.
    
        Diferencia el centro y el anillo exterior para separar monedas bimetálicas, 
        de cobre y doradas, seleccionando la opción con menor diferencia de diámetro.
    """
    dif_saturacion = abs(int(hsv_centro[1]) - int(hsv_anillo[1]))
    es_bimetal = dif_saturacion > 40
    
    es_cobre = hsv_centro[0] < 15 or hsv_anillo[0] < 15

    if es_bimetal:
        diccionario_valido = monedas_bimetal
    elif es_cobre:
        diccionario_valido = monedas_cobre
    else:
        diccionario_valido = monedas_doradas

    moneda = None
    min_diferencia = float('inf')
    for valor, diametro in diccionario_valido.items():
        dif = abs(diametro - diametro_medido)
        if dif < min_diferencia:
            min_diferencia = dif
            moneda = valor
            
    if min_diferencia > 2.5:
        return None
            
    return moneda

escala = None
punto_clic = None
ultimos_totales = [] # para estabilizar

def detectar_clic(evento, x, y, flags, param):
    """Guarda las coordenadas al hacer clic izquierdo."""
    global punto_clic
    if evento == cv2.EVENT_LBUTTONDOWN:
        punto_clic = (x, y)

vid = cv2.VideoCapture(0)
cv2.namedWindow('Cam')
cv2.setMouseCallback('Cam', detectar_clic)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

while True:
    ret, frame = vid.read()
    if not ret: break

    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gris_clahe = clahe.apply(gris)
    gris_suavizado = cv2.medianBlur(gris_clahe, 7)
    
    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Fuente: https://www.luisllamas.es/opencv-transformada-hough-lineas-circulos/
    circulos = cv2.HoughCircles(
        gris_suavizado, cv2.HOUGH_GRADIENT, 1, 30,
        param1=100, param2=40, minRadius=15, maxRadius=80
    )

    total_frame = 0.0

    if circulos is not None:
        circulos = np.round(circulos[0, :]).astype("int") # Coordenadas a enteros

        if punto_clic is not None and escala is None:
            for (x, y, r) in circulos:
                if (punto_clic[0] - x)**2 + (punto_clic[1] - y)**2 <= r**2:
                    escala = 23.25 / (r * 2)
                    print(f"Calibrada escala: {escala:.4f}")
                    punto_clic = None
                    break

        if escala is None:
            cv2.putText(frame, "Haz clic en la moneda de 1€", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        else:
            cv2.putText(frame, "Pulsa 'R' para reiniciar la escala", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
                        
            for (x, y, r) in circulos:
                diametro_px = r * 2
                diametro_mm = diametro_px * escala
                
                if diametro_mm > 14.0:
                    # Centro
                    mask_centro = np.zeros(frame.shape[:2], dtype=np.uint8)
                    cv2.circle(mask_centro, (x, y), int(r * 0.4), 255, -1)
                    hsv_centro = cv2.mean(hsv_frame, mask=mask_centro)[:3]

                    # Aro exterior
                    mask_borde = np.zeros(frame.shape[:2], dtype=np.uint8)
                    cv2.circle(mask_borde, (x, y), int(r * 0.85), 255, -1)
                    cv2.circle(mask_borde, (x, y), int(r * 0.50), 0, -1)
                    hsv_borde = cv2.mean(hsv_frame, mask=mask_borde)[:3]

                    valor = clasificar_moneda_hsv(diametro_mm, hsv_centro, hsv_borde)
                    
                    if valor is not None:
                        total_frame += valor
                        cv2.circle(frame, (x, y), r, (0, 255, 0), 2)
                        cv2.putText(frame, f"{valor}€", (x - 20, y + 5), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    if escala is not None:
        total_frame = round(total_frame, 2)
        ultimos_totales.append(total_frame)
        
        if len(ultimos_totales) > 15: # efecto ventana
            ultimos_totales.pop(0)
        
        if len(ultimos_totales) > 0:
            total = max(set(ultimos_totales), key=ultimos_totales.count) # moda
        else:
            total = 0.0

        cv2.putText(frame, f"TOTAL = {total:.2f} €", (20, frame.shape[0] - 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)

    cv2.imshow('Cam', frame)

    tecla = cv2.waitKey(20) & 0xFF
    if tecla == 27: 
        break
    elif tecla in [ord('r'), ord('R')]: 
        escala = None 
        punto_clic = None
        ultimos_totales.clear()
        print("Escala reiniciada")

vid.release()
cv2.destroyAllWindows()

---

## Tarea 2: Segmentación y clasificación de microplásticos

- Aplica técnicas de umbralizado mediante OpenCV para segmentar muestras de microplásticos en imágenes reales y desarrolla un clasificador basado en características geométricas y visuales que permita distinguir los distintos tipos de partículas.

---